# 01 — Datenexploration und Preprocessing

Ziel: finale Datensatzstruktur, Klassenverteilung und Kategorieverteilung untersuchen. Preprocessing, Labelregeln und Data-Quality-Checks dokumentieren.

**Labelregeln:**
- `edible`: sicher essbar, darf optisch unperfekt sein.
- `non_edible`: Schimmel, Schleim, deutliche Fäule oder starke Verderbnis.

### Aufbau
1. Datenquellen
2. Gesamtverteilung
3. Kategorieverteilung
4. Schwache Kategorien
5. Data-Quality-Checks
6. Dataset-Rebuild-Logik
7. Beispielbilder & Resize
8. Optionales Batch-Preprocessing

In [ ]:
from pathlib import Path
import json
import math
import hashlib
import re
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, UnidentifiedImageError

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    pass

import os
os.chdir("..")

PROJECT_ROOT = Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
TRAIN_DIR    = DATA_DIR / "train"
VAL_DIR      = DATA_DIR / "val"
REPORTS_DIR  = PROJECT_ROOT / "reports"
MODELS_DIR   = PROJECT_ROOT / "models"
FINAL_MODEL  = MODELS_DIR / "freshify_baseline_with_new_raw.keras"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic"}
CATEGORIES = ["banane", "erdbeere", "gurke", "orange", "paprika", "zitrone"]

plt.style.use("default")
pd.set_option("display.max_colwidth", 120)

## 1. Datenquellen

Alle verwendeten externen Datensätze und Quellen:

In [ ]:
pd.DataFrame([
    {"Quelle": "https://www.kaggle.com/datasets/ulnnproject/food-freshness-dataset",
     "Verwendung": "Basisdatensatz für essbare und nicht essbare Lebensmittel"},
    {"Quelle": "https://www.kaggle.com/datasets/yusufemir/lemon-quality-dataset",
     "Verwendung": "Ergänzung für Zitrone"},
    {"Quelle": "https://www.kaggle.com/datasets/abdulbasit31/strawberry-dataset",
     "Verwendung": "Ergänzung für Erdbeere"},
    {"Quelle": "https://www.kaggle.com/datasets/abdulrafeyyashir/fresh-vs-rotten-fruit-images",
     "Verwendung": "Ergänzende frische/verdorbene Obstbilder"},
    {"Quelle": "https://www.kaggle.com/datasets/muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten",
     "Verwendung": "Ergänzung für Gurke/Paprika und weitere Fresh-vs-Rotten-Beispiele"},
    {"Quelle": "Pexels",
     "Verwendung": "Einzelne ergänzende Bilder für edible/zitrone"},
    {"Quelle": "Eigene Smartphonebilder",
     "Verwendung": "Zusätzliche realistischere non_edible Beispiele, inzwischen in data/train/ übernommen"},
])

In [ ]:
def image_files(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def infer_label(path):
    parts = [p.lower() for p in Path(path).parts]
    if "non_edible" in parts:
        return "non_edible"
    if "edible" in parts:
        return "edible"
    return "unknown"

def infer_category(path):
    text = str(path).lower()
    aliases = {
        "banane":   ["banane", "banana"],
        "erdbeere": ["erdbeere", "strawberry"],
        "gurke":    ["gurke", "cucumber"],
        "orange":   ["orange"],
        "paprika":  ["paprika", "pepper", "bell_pepper"],
        "zitrone":  ["zitrone", "lemon"],
    }
    for category, keys in aliases.items():
        if any(k in text for k in keys):
            return category
    return "unknown"

records = []
for split, split_dir in [("train", TRAIN_DIR), ("val", VAL_DIR)]:
    for path in image_files(split_dir):
        records.append({
            "split":    split,
            "label":    infer_label(path),
            "category": infer_category(path),
            "path":     str(path.relative_to(PROJECT_ROOT)),
        })

images_df = pd.DataFrame(records)
images_df.head()

## 2. Gesamtverteilung pro Split

`data/val/` ist klein; einzelne Fehlklassifikationen verschieben Accuracy, Precision und Recall sichtbar.

In [ ]:
summary = (
    images_df.groupby(["split", "label"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "label"])
) if not images_df.empty else pd.DataFrame(columns=["split", "label", "count"])
summary

In [ ]:
if not summary.empty:
    pivot = summary.pivot(index="split", columns="label", values="count").fillna(0)
    ax = pivot.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Train/Val-Verteilung nach Label")
    ax.set_xlabel("Split")
    ax.set_ylabel("Anzahl Bilder")
    ax.legend(title="Label")
    plt.tight_layout()
    plt.show()

## 3. Kategorieverteilung pro Split, Label und Kategorie

Abgedeckte Kategorien: Banane, Erdbeere, Gurke, Orange, Paprika, Zitrone.

In [ ]:
category_summary = (
    images_df.groupby(["split", "label", "category"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["split", "label", "category"])
) if not images_df.empty else pd.DataFrame(columns=["split", "label", "category", "count"])
category_summary

In [ ]:
for split in ["train", "val"]:
    data = category_summary[category_summary["split"] == split]
    if data.empty:
        continue
    pivot = data.pivot_table(index="category", columns="label", values="count", fill_value=0)
    ax = pivot.reindex(CATEGORIES + ["unknown"]).dropna(how="all").plot(kind="bar", figsize=(9, 4))
    ax.set_title(f"Kategorieverteilung pro Label: {split}")
    ax.set_xlabel("Kategorie")
    ax.set_ylabel("Anzahl Bilder")
    ax.legend(title="Label")
    plt.tight_layout()
    plt.show()

## 4. Schwache Kategorien

Kategorien mit kleiner Datenmenge, kleiner Val-Menge oder Hinweisen aus `reports/training_findings.md`.

In [ ]:
findings_path = REPORTS_DIR / "training_findings.md"
findings_text = findings_path.read_text(encoding="utf-8", errors="replace") if findings_path.exists() else ""

weak_rows = []
if not category_summary.empty:
    pivot = category_summary.pivot_table(
        index=["label", "category"], columns="split", values="count", fill_value=0
    )
    for (label, category), row in pivot.iterrows():
        train_count = int(row.get("train", 0))
        val_count   = int(row.get("val", 0))
        reasons = []
        if train_count < 30:
            reasons.append("kleine Datenmenge")
        if val_count < 5:
            reasons.append("kleine Val-Menge")
        if category != "unknown" and category.lower() in findings_text.lower():
            reasons.append("Hinweis in training_findings.md")
        if reasons:
            weak_rows.append({
                "label":       label,
                "category":    category,
                "train_count": train_count,
                "val_count":   val_count,
                "hinweise":    "; ".join(reasons),
            })

pd.DataFrame(weak_rows)

## 5. Data-Quality-Checks

- nicht-lesbare Bilder
- Nicht-Bild-Dateien
- exakte Duplikate (SHA-256)
- Train-Val-Duplikate

In [ ]:
all_files = []
for split, split_dir in [("train", TRAIN_DIR), ("val", VAL_DIR)]:
    if split_dir.exists():
        for path in sorted(p for p in split_dir.rglob("*") if p.is_file()):
            all_files.append((split, path))

unreadable = []
non_images = []
hashes     = defaultdict(list)

for split, path in all_files:
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        non_images.append(str(path.relative_to(PROJECT_ROOT)))
        continue
    try:
        with Image.open(path) as img:
            img.verify()
        digest = hashlib.md5(path.read_bytes()).hexdigest()
        hashes[digest].append((split, str(path.relative_to(PROJECT_ROOT))))
    except (UnidentifiedImageError, OSError, ValueError):
        unreadable.append(str(path.relative_to(PROJECT_ROOT)))

exact_duplicates    = {d: items for d, items in hashes.items() if len(items) > 1}
train_val_duplicates = {
    d: items for d, items in exact_duplicates.items()
    if {"train", "val"}.issubset({s for s, _ in items})
}

pd.DataFrame([
    {"check": "nicht-lesbare Bilder",  "count": len(unreadable),        "examples": unreadable[:5]},
    {"check": "Nicht-Bild-Dateien",    "count": len(non_images),        "examples": non_images[:5]},
    {"check": "exakte Duplikate",      "count": len(exact_duplicates),  "examples": list(exact_duplicates.values())[:3]},
    {"check": "Train-Val-Duplikate",   "count": len(train_val_duplicates), "examples": list(train_val_duplicates.values())[:3]},
])

## 6. Dataset-Rebuild-Logik aus `prepare_dataset.py`

Der echte Rebuild erfolgt nur per CLI — `python src/prepare_dataset.py`. Dieser Block dokumentiert die Schritte, ohne `data/` zu verändern.

In [ ]:
dataset_builder_path = PROJECT_ROOT / "src" / "prepare_dataset.py"
config_path          = PROJECT_ROOT / "src" / "config.py"

print(f"Dataset-Builder vorhanden: {dataset_builder_path.exists()} ({dataset_builder_path})")
print(f"Config vorhanden:          {config_path.exists()} ({config_path})")

pd.DataFrame([
    {"Schritt": "Quellen sammeln",
     "Beschreibung": "structured_sources, raw_sources und background_aug_dir werden gesammelt."},
    {"Schritt": "Kategorie ableiten",
     "Beschreibung": "Kategorie kommt aus Dateiname oder Elternordner über CATEGORY_ALIASES."},
    {"Schritt": "Hash berechnen",
     "Beschreibung": "SHA-256 für exakte Duplikate, Average Hash für Near-Duplicates."},
    {"Schritt": "Deduplikation",
     "Beschreibung": "Bei gleichem SHA-256 gewinnt structured vor raw vor background_aug."},
    {"Schritt": "Split-Zuweisung",
     "Beschreibung": "Pro Label/Kategorie werden Near-Duplicate-Cluster gemeinsam Train oder Val zugeordnet."},
    {"Schritt": "Background-Aug",
     "Beschreibung": "Background-Augmented Bilder werden immer train zugeordnet."},
    {"Schritt": "Manifest",
     "Beschreibung": "reports/dataset_manifest.csv dokumentiert split, label, category, source_type, source_path, output_path, sha256, average_hash."},
])

In [ ]:
manifest_path = REPORTS_DIR / "dataset_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    display(manifest.head())
    display(
        manifest.groupby(["split", "label", "category", "source_type"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["split", "label", "category", "source_type"])
    )
else:
    print(f"Manifest fehlt: {manifest_path}")

## 7. Beispielbilder & Resize auf 224 × 224

Das Modell (`train.py`) lädt alle Bilder über `image_dataset_from_directory` mit `image_size=(224, 224)`, was einem direkten Resize entspricht.

In [ ]:
def show_sample_grid(df, max_per_row=6):
    if df.empty:
        print("Keine Bilddaten gefunden.")
        return
    sample_rows = []
    for label in ["edible", "non_edible"]:
        for category in CATEGORIES:
            subset = df[(df["label"] == label) & (df["category"] == category)]
            if not subset.empty:
                sample_rows.append(subset.iloc[0])
    if not sample_rows:
        print("Keine passenden Beispielbilder gefunden.")
        return
    n    = len(sample_rows)
    cols = min(max_per_row, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.4, rows * 2.7))
    axes = np.array(axes).reshape(-1)
    for ax, row in zip(axes, sample_rows):
        path = PROJECT_ROOT / row["path"]
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(f'{row["label"]}\n{row["category"]}', fontsize=9)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle("Beispielbilder pro Kategorie und Label")
    plt.tight_layout()
    plt.show()

show_sample_grid(images_df)

In [ ]:
if not images_df.empty:
    sample_path = PROJECT_ROOT / images_df.iloc[0]["path"]
    original    = Image.open(sample_path).convert("RGB")
    resized     = original.resize((224, 224))
    fig, axes   = plt.subplots(1, 2, figsize=(7, 3.5))
    axes[0].imshow(original)
    axes[0].set_title(f"Original: {original.size[0]} × {original.size[1]}")
    axes[1].imshow(resized)
    axes[1].set_title("Resize: 224 × 224")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Kein Beispielbild verfügbar.")

## 8. Optionales Batch-Preprocessing externer Rohbilder

Utility für externe Rohbilder (z. B. Smartphonebilder). Standardmäßig `DRY_RUN=True` — Pfade erst bewusst setzen und `DRY_RUN=False` wählen, bevor Dateien verändert werden.

Der Code entspricht dem Skript, das für die `new_raw`-Bilder in `data/train/` verwendet wurde.

In [ ]:
from tqdm import tqdm

RAW_SRC        = Path("/mnt/g/FAU/ML4B/input_images_ganz_große_dateien/")
RAW_DST        = Path("/mnt/g/FAU/ML4B/output_224/")
SIZE           = 224
QUALITY        = 85
RAW_EXTENSIONS = {".jpg", ".jpeg", ".png", ".heic"}
DRY_RUN        = True

def image_resize_save(src_path: Path, dst_path: Path, size=SIZE, quality=QUALITY):
    """Resize auf size×size (center-crop nach proportionalem Scale), speichert als JPEG."""
    with Image.open(src_path) as im:
        im    = ImageOps.exif_transpose(im).convert("RGB")
        w, h  = im.size
        scale = size / min(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        im    = im.resize((new_w, new_h), Image.LANCZOS)
        im    = ImageOps.fit(im, (size, size), method=Image.LANCZOS, centering=(0.5, 0.5))
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst_path, format="JPEG", quality=quality, optimize=True, progressive=True)

def preprocess_raw_images(src=RAW_SRC, dst=RAW_DST, dry_run=DRY_RUN):
    files = [p for p in Path(src).rglob("*") if p.is_file() and p.suffix.lower() in RAW_EXTENSIONS]
    print(f"{len(files)} Dateien gefunden.")
    if dry_run:
        return pd.DataFrame({
            "src": [str(p) for p in files[:10]],
            "dst": [str(Path(dst) / p.relative_to(src).with_suffix(".jpg")) for p in files[:10]],
        })
    for p in tqdm(files, desc="Processing"):
        out = Path(dst) / p.relative_to(src).with_suffix(".jpg")
        try:
            image_resize_save(p, out)
        except Exception as exc:
            print("ERROR", p, exc)

preprocess_raw_images()

## Optionaler sicherer Import vorsortierter Bilder

Kopiert (kein Move) vorsortierte Bilder nach `data/train/edible/` mit fortlaufender Nummerierung. `IMPORT_DRY_RUN=True` zeigt nur den Plan.

In [ ]:
SORTED_SRC       = Path("/mnt/g/FAU/ML4B/sorted/")
TRAIN_EDIBLE_DST = TRAIN_DIR / "edible"
IMPORT_DRY_RUN   = True
MOVE_FILES       = False

def next_numeric_counter(dst: Path):
    if not dst.exists():
        return 1
    nums = [int(m.group()) for f in dst.iterdir() if f.is_file() and (m := re.search(r"\d+", f.name))]
    return max(nums, default=0) + 1

def numbered_name(original_name: str, counter: int):
    stem   = Path(original_name).stem
    suffix = Path(original_name).suffix.lower()
    stem   = re.sub(r"\d+", str(counter), stem, count=1) if re.search(r"\d+", stem) else f"{stem}_{counter}"
    return f"{stem}{suffix}"

def import_sorted_images(src=SORTED_SRC, dst=TRAIN_EDIBLE_DST, dry_run=IMPORT_DRY_RUN, move_files=MOVE_FILES):
    src, dst = Path(src), Path(dst)
    files    = sorted(p for p in src.iterdir() if p.is_file()) if src.exists() else []
    counter  = next_numeric_counter(dst)
    plan     = []
    for p in files:
        target = dst / numbered_name(p.name, counter)
        plan.append({"src": str(p), "dst": str(target), "action": "move" if move_files else "copy"})
        counter += 1
    if dry_run:
        return pd.DataFrame(plan)
    dst.mkdir(parents=True, exist_ok=True)
    for item in plan:
        (shutil.move if move_files else shutil.copy2)(item["src"], item["dst"])
    return pd.DataFrame(plan)

import_sorted_images()

### Interpretation

- **Val-Set klein:** `data/val/` enthält 62 Bilder — einzelne Fehlklassifikationen verschieben Accuracy, Recall und FN sichtbar.
- **`new_raw` kein Holdout mehr:** Die Smartphonebilder sind inzwischen Teil von `data/train/`. Für belastbare Real-World-Metriken wird ein neuer unabhängiger Holdout mit `edible` und `non_edible` benötigt.
- **Priorität Datenerweiterung:** Kategorien mit kleiner Train-Menge, kleiner Val-Menge oder auffälligen Fehlerhinweisen sollten priorisiert erweitert werden.